In [5]:
# ==========================================================
# Import Libraries
# ==========================================================

import warnings
warnings.filterwarnings("ignore")

import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler,
    OneHotEncoder
)

from sklearn.compose import ColumnTransformer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier

print("="*60)
print("Libraries Imported Successfully")
print("="*60)

Libraries Imported Successfully


In [6]:
# Load the Dataset
FILE_PATH = '/content/sample_data'

df = pd.read_csv(r"C:\Users\Admin\Desktop\diabetes_dataset.csv")

print(f'Dataset loaded successfully.')
print(f'  Rows    : {df.shape[0]:,}')
print(f'  Columns : {df.shape[1]}')
print()

# Display first 5 rows to confirm correct loading
print('First 5 Rows ')
df.head()

Dataset loaded successfully.
  Rows    : 100,000
  Columns : 31

First 5 Rows 


,age,gender,ethnicity,education_level,income_level,employment_status,smoking_status,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,...,hdl_cholesterol,ldl_cholesterol,triglycerides,glucose_fasting,glucose_postprandial,insulin_level,hba1c,diabetes_risk_score,diabetes_stage,diagnosed_diabetes
0,58,Male,Asian,Highschool,Lower-Middle,Employed,Never,0,215,5.7,...,41,160,145,136,236,6.36,8.18,29.6,Type 2,1
1,48,Female,White,Highschool,Middle,Employed,Former,1,143,6.7,...,55,50,30,93,150,2.00,5.63,23.0,No Diabetes,0
2,60,Male,Hispanic,Highschool,Middle,Unemployed,Never,1,57,6.4,...,66,99,36,118,195,5.07,7.51,44.7,Type 2,1
3,74,Female,Black,Highschool,Low,Retired,Never,0,49,3.4,...,50,79,140,139,253,5.28,9.03,38.2,Type 2,1
4,46,Male,White,Graduate,Middle,Retired,Never,1,109,7.2,...,52,125,160,137,184,12.74,7.20,23.5,Type 2,1


In [7]:
# ==========================================================
# Encode Target
# ==========================================================

label_encoder = LabelEncoder()

df["diabetes_stage"] = label_encoder.fit_transform(
    df["diabetes_stage"]
)

print("="*60)
print("Target Classes")
print("="*60)

for i, cls in enumerate(label_encoder.classes_):
    print(f"{i} ---> {cls}")

Target Classes
0 ---> Gestational
1 ---> No Diabetes
2 ---> Pre-Diabetes
3 ---> Type 1
4 ---> Type 2


In [9]:
# ==========================================================
# Deployment Features
# ==========================================================

TARGET_COL = "diabetes_stage"

SELECTED_FEATURES = [

    "age",
    "gender",

    "bmi",

    "hba1c",

    "glucose_fasting",
    "glucose_postprandial",

    "cholesterol_total",

    "systolic_bp",

    "family_history_diabetes",

    "physical_activity_minutes_per_week",

    "insulin_level"

]

X = df[SELECTED_FEATURES]

y = df[TARGET_COL]

print("="*60)
print("Selected Features")
print("="*60)

print(X.columns.tolist())

Selected Features
['age', 'gender', 'bmi', 'hba1c', 'glucose_fasting', 'glucose_postprandial', 'cholesterol_total', 'systolic_bp', 'family_history_diabetes', 'physical_activity_minutes_per_week', 'insulin_level']


In [10]:
# ==========================================================
# Train / Validation / Test Split
# ==========================================================

X_temp, X_test, y_temp, y_test = train_test_split(

    X,
    y,

    test_size=0.05,

    random_state=42,

    stratify=y

)

val_ratio = 0.15 / 0.95

X_train, X_val, y_train, y_val = train_test_split(

    X_temp,
    y_temp,

    test_size=val_ratio,

    random_state=42,

    stratify=y_temp

)

print("="*60)
print("Data Split Completed")
print("="*60)

print("Train :", X_train.shape)
print("Validation :", X_val.shape)
print("Test :", X_test.shape)

Data Split Completed
Train : (80000, 11)
Validation : (15000, 11)
Test : (5000, 11)


In [11]:
# ==========================================================
# Preprocessing
# ==========================================================

categorical_features = [

    "gender"

]

numeric_features = [

    "age",

    "bmi",

    "hba1c",

    "glucose_fasting",

    "glucose_postprandial",

    "cholesterol_total",

    "systolic_bp",

    "family_history_diabetes",

    "physical_activity_minutes_per_week",

    "insulin_level"

]

preprocessor = ColumnTransformer(

    transformers=[

        (

            "num",

            StandardScaler(),

            numeric_features

        ),

        (

            "cat",

            OneHotEncoder(

                drop="first",

                handle_unknown="ignore"

            ),

            categorical_features

        )

    ]

)

X_train = preprocessor.fit_transform(X_train)

X_val = preprocessor.transform(X_val)

X_test = preprocessor.transform(X_test)

print("="*60)
print("Preprocessing Completed")
print("="*60)

print(X_train.shape)

Preprocessing Completed
(80000, 12)


In [19]:
import joblib
import os

os.makedirs("saved_models", exist_ok=True)

joblib.dump(preprocessor, "saved_models/preprocessor.pkl")

print("Preprocessor Saved Successfully")

Preprocessor Saved Successfully


In [13]:
# ==========================================================
# Train XGBoost Model
# ==========================================================

xgb_model = XGBClassifier(

    objective="multi:softprob",

    num_class=len(np.unique(y_train)),

    n_estimators=300,

    max_depth=6,

    learning_rate=0.05,

    subsample=0.8,

    colsample_bytree=0.8,

    eval_metric="mlogloss",

    random_state=42

)

print("="*60)
print("Training XGBoost...")
print("="*60)

xgb_model.fit(
    X_train,
    y_train
)

print("Training Completed Successfully!")

Training XGBoost...
Training Completed Successfully!


In [14]:
# ==========================================================
# Model Evaluation
# ==========================================================

val_pred = xgb_model.predict(X_val)

val_prob = xgb_model.predict_proba(X_val)

accuracy = accuracy_score(y_val, val_pred)

precision = precision_score(
    y_val,
    val_pred,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_val,
    val_pred,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_val,
    val_pred,
    average="weighted",
    zero_division=0
)

roc_auc = roc_auc_score(
    y_val,
    val_prob,
    multi_class="ovr",
    average="weighted"
)

print("="*60)
print("DEPLOYMENT MODEL RESULTS")
print("="*60)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC AUC  : {roc_auc:.4f}")

print("\nClassification Report\n")

print(

    classification_report(

        y_val,

        val_pred,

        target_names=label_encoder.classes_,

        zero_division=0

    )

)

DEPLOYMENT MODEL RESULTS
Accuracy : 0.9161
Precision: 0.9255
Recall   : 0.9161
F1 Score : 0.9151
ROC AUC  : 0.9523

Classification Report

              precision    recall  f1-score   support

 Gestational       0.00      0.00      0.00        42
 No Diabetes       0.85      0.99      0.91      1197
Pre-Diabetes       0.82      1.00      0.90      4777
      Type 1       0.00      0.00      0.00        18
      Type 2       1.00      0.87      0.93      8966

    accuracy                           0.92     15000
   macro avg       0.53      0.57      0.55     15000
weighted avg       0.93      0.92      0.92     15000



In [17]:
# ==========================================================
# Save Deployment Model
# ==========================================================

import os
import joblib

os.makedirs("saved_models", exist_ok=True)

# Save Model
joblib.dump(
    xgb_model,
    "saved_models/xgb_model.pkl"
)

# Save Preprocessor
joblib.dump(
    preprocessor,
    "saved_models/preprocessor.pkl"
)

# Save Label Encoder
joblib.dump(
    label_encoder,
    "saved_models/label_encoder.pkl"
)

print("="*60)
print("Deployment Model Saved Successfully")
print("="*60)

Deployment Model Saved Successfully


In [18]:
# ==========================================================
# Test Saved Model
# ==========================================================

import joblib

model = joblib.load("saved_models/xgb_model.pkl")

preprocessor = joblib.load("saved_models/preprocessor.pkl")

label_encoder = joblib.load("saved_models/label_encoder.pkl")

print("✅ Model Loaded Successfully")

print("✅ Preprocessor Loaded Successfully")

print("✅ Label Encoder Loaded Successfully")

✅ Model Loaded Successfully
✅ Preprocessor Loaded Successfully
✅ Label Encoder Loaded Successfully


In [20]:
joblib.load("saved_models/preprocessor.pkl")

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``